# Bài 3
Đây là notebook chứa mã nguồn đầy đủ của bài 3.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import pulp
import pyomo.environ as pyo
from scipy.optimize import linprog, minimize, milp, LinearConstraint, Bounds
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize as pymoo_minimize

from src.data_loader import get_data


In [ ]:
def solve_bai03(data_dir=None, w_growth=0.15, w_productivity=0.15, w_spillover=0.20,
                w_export=0.15, w_employment=0.10, w_ai=0.20, w_risk=0.15):
    sectors = ['Nông nghiệp', 'Chế biến', 'Xây dựng', 'Khai khoáng', 'Bán lẻ',
               'Tài chính', 'Logistics', 'CNTT', 'Giáo dục', 'Y tế']
    col_names = ['Tăng trưởng', 'Năng suất', 'Lan tỏa', 'Xuất khẩu', 'Việc làm', 'AI Readiness', 'Rủi ro TĐH']
    # [growth%, productivity, spillover, export, labor(M), ai_readiness, automation_risk]
    X = np.array([
        [3.27,  103,  0.35,  40.5, 13.2, 15, 18],
        [9.64,  241,  0.78, 290.9, 11.5, 55, 42],
        [7.45,  169,  0.42,   2.5,  4.8, 20, 25],
        [-1.2, 1290,  0.30,   8.2,  0.3, 30, 55],
        [7.10,  145,  0.55,   5.5,  7.8, 48, 38],
        [7.36, 1072,  0.85,   1.2, 0.55, 72, 52],
        [9.93,  321,  0.72,   3.1, 1.95, 42, 35],
        [7.85,  714,  0.92,  178,  0.62, 88, 28],
        [6.42,  206,  0.65,   0,   2.15, 38, 22],
        [6.85,  437,  0.60,   0,   0.75, 45, 18],
    ], float)

    good = X[:, :6]
    bad  = X[:, 6]

    # === (1) Min-max normalization (đảo dấu Risk) ===
    Gn = (good - good.min(0)) / (np.ptp(good, axis=0) + 1e-9)
    R  = (bad - bad.min()) / (np.ptp(bad) + 1e-9)
    # Ma trận chuẩn hóa đầy đủ 7 cột (Risk đã đảo dấu: 1 - R)
    norm_matrix = np.column_stack([Gn, 1 - R])
    norm_matrix_list = [[round(float(v), 4) for v in row] for row in norm_matrix]

    # === (2) Tính Priority với trọng số mặc định ===
    w = np.array([w_growth, w_productivity, w_spillover, w_export, w_employment, w_ai], float)
    wr = w_risk

    score = Gn @ w - wr * R
    idx = np.argsort(-score)

    result = []
    for i in idx:
        result.append({
            'sector_name_vi': sectors[i],
            'Priority': round(float(score[i]), 4),
            'rank': int(np.where(idx == i)[0][0]) + 1,
        })

    # === (3) Phân tích độ nhạy: a₆ từ 0.05 đến 0.40 ===
    a6_values = [round(v, 2) for v in np.arange(0.05, 0.45, 0.05)]
    base_other = np.array([w_growth, w_productivity, w_spillover, w_export, w_employment], float)
    sensitivity_heatmap = []  # rows = sectors, cols = a6 values
    sensitivity_top3 = {}
    for a6 in a6_values:
        # Chuẩn hóa lại tổng = 1: scale other weights proportionally
        remaining = 1.0 - a6 - wr
        other_sum = base_other.sum() or 1
        w_scaled = base_other * (remaining / other_sum)
        s_score = Gn @ np.append(w_scaled, a6) - wr * R
        s_idx = np.argsort(-s_score)
        sensitivity_top3[str(a6)] = [sectors[i] for i in s_idx[:3]]
        sensitivity_heatmap.append([round(float(s_score[i]), 4) for i in range(10)])

    # Transpose: rows=sectors, cols=a6_values
    heatmap_data = [[sensitivity_heatmap[j][i] for j in range(len(a6_values))] for i in range(10)]

    # === (4) So sánh 2 bộ trọng số (Dynamic) ===
    # "Định hướng tăng trưởng": nhân đôi trọng số Tăng trưởng, Năng suất, Xuất khẩu từ base của user
    w_growth_oriented = np.copy(w)
    w_growth_oriented[0] *= 2.0 # growth
    w_growth_oriented[1] *= 2.0 # productivity
    w_growth_oriented[3] *= 2.0 # export
    w_growth_oriented = w_growth_oriented / (w_growth_oriented.sum() + 1e-9) * (1 - wr)
    
    score_growth = Gn @ w_growth_oriented - wr * R
    idx_growth = np.argsort(-score_growth)

    # "Định hướng bao trùm": nhân đôi trọng số Việc làm, Lan tỏa, AI từ base của user
    w_inclusive = np.copy(w)
    w_inclusive[2] *= 2.0 # spillover
    w_inclusive[4] *= 2.0 # employment
    w_inclusive[5] *= 2.0 # ai
    w_inclusive = w_inclusive / (w_inclusive.sum() + 1e-9) * (1 - wr)
    
    score_inclusive = Gn @ w_inclusive - wr * R
    idx_inclusive = np.argsort(-score_inclusive)

    scenario_comparison = {
        'growth': {
            'top3': [sectors[i] for i in idx_growth[:3]],
            'scores': {sectors[i]: round(float(score_growth[i]), 4) for i in range(10)},
        },
        'inclusive': {
            'top3': [sectors[i] for i in idx_inclusive[:3]],
            'scores': {sectors[i]: round(float(score_inclusive[i]), 4) for i in range(10)},
        },
    }

    return {
        'ranking': result,
        'sectors': sectors,
        'col_names': col_names,
        'norm_matrix': norm_matrix_list,
        'a6_values': [str(v) for v in a6_values],
        'heatmap_data': heatmap_data,
        'sensitivity_top3': sensitivity_top3,
        'scenario_comparison': scenario_comparison,
    }

In [ ]:
if __name__ == '__main__':
    res = solve_bai03()
    # In ra một số key để kiểm tra
    if isinstance(res, dict):
        print(res.keys())